# Figure 2 - Marker Gene comparison

In [ ]:
from pathlib import Path
import scanpy as sc
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from tqdm import tqdm
from pprint import pprint

import importlib
import scatlastb_utils as atl

sc.set_figure_params(frameon=False, dpi=80, fontsize=12)
plt.rcParams["svg.fonttype"] = "none"

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from misc import clean_prefixes

In [ ]:
integrations = [
    'HLCAv1',
    'highQC',
]

In [ ]:
figure_dir = Path('figures/figure2/C')
figure_dir.mkdir(exist_ok=True, parents=True)

## Read data

In [ ]:
adata = atl.io.read_anndata(
    'data/pipeline/HLCAv1/QC/marker_genes/dataset~HLCAv1_extended/file_id~majority_voting:collect:HLCAv1_extended.zarr',
    X='X',
    obs='obs',
    var='var',
    obsm='obsm',
    uns='uns',
    obsp='obsp',
    dask_slots=['layers', 'X', 'obsp'],
    dask=True,
    backed=True,
)
clean_prefixes(adata, 'label_transfer:clustering:')
adata.obs['n_genes'] = adata.obs['n_genes'].astype('float32')
adata

In [ ]:
# set masks
extended_only_mask = adata.obs['core_or_extension'] == 'extension'
core_only = adata.obs['core_or_extension'] == 'core'
highQC_reference = core_only & (adata.obs['qc_status'] == 'passed')

In [ ]:
# set colors
ref_key = 'ann_finest_level'
for col in tqdm([x for x in adata.obs.columns if x.startswith('majority_reference')]):
    adata.obs[col] = adata.obs[col].cat.set_categories(adata.obs[ref_key].cat.categories)

## Fig 2C: Marker gene overlap

In [ ]:
import traceback
from marker_genes_functions import get_marker_genes
from misc import run_similarity_analysis

In [ ]:
n_genes = 50
marker_genes_HLCAv1 = get_marker_genes(
    adata,
    key='marker_genes_group=majority_reference--HLCAv1',
    n_genes=n_genes,
)

marker_genes_highqc = get_marker_genes(
    adata,
    key='marker_genes_group=majority_reference--highQC',
    n_genes=n_genes,
)

In [ ]:
run_similarity_analysis(
    group_col='group',
    value_col='gene',
    df1=marker_genes_highqc,
    df1_label='High QC',
    df2=marker_genes_HLCAv1,
    df2_label='HLCAv1',
    title='Cell type marker gene overlap',
    figsize=(14, 14),
    cmap='viridis',
    lower_only=True,
    self_compare=True,
    cbar_name='Jaccard overlap',
    save_dir=figure_dir,
)

### Ext Fig 2A: Per lineage marker overlap

In [ ]:
lineage_key = 'ann_level_1'
anno_map = adata.obs[[lineage_key, 'ann_finest_level']].value_counts()

for lineage in anno_map.index.get_level_values(lineage_key).unique():
    groups = anno_map.xs(key=lineage, level=lineage_key).index.values
    if len(groups) <= 1:
        print('not enough cell types per group, skipping...')
    try:
        save_dir = figure_dir / lineage
        save_dir.mkdir(exist_ok=True, parents=True)
        _ = run_similarity_analysis(
            df1=marker_genes_highqc.query('group.isin(@groups)'),
            df1_label='High QC',
            df2=marker_genes_HLCAv1.query('group.isin(@groups)'),
            df2_label='HLCAv1',
            group_col='group',
            value_col='gene',
            cmap='viridis',
            title=f'Cell type marker gene overlap for {lineage=}',
            self_compare=True,
            figsize=(7,7),
            title_fontsize=20,
            cbar_name='Jaccard overlap',
            save_dir=save_dir,
        )
    except:
        print('skipping:', lineage)
        print(f'{groups=}')
        traceback.print_exc()

## Extended Fig. 2B: Check overlap with curated marker genes

In [ ]:
import yaml
from misc import back_to_back_barplot

In [ ]:
with open('configs/HLCA/gene_sets_full.yaml', 'r') as f:
    marker_genes_dict = yaml.safe_load(f)

In [ ]:
for gene_set in marker_genes_dict.values():
    for gene in gene_set:
        if gene not in adata.var['feature_name'].values:
            gene_set.remove(gene)

In [ ]:
def get_frac_overlap(d1, d2):
    data = []
    for k in d1.keys() & d2.keys():
        s1, s2 = set(d1[k]), set(d2[k])
        data.append({
            'cell_type': k,
            'overlap': len(s1 & s2) / len(s1),
            'present': list(s1 & s2),
            'missing': list(s1 - s2)
        })

    return pd.DataFrame(data)

In [ ]:
marker_overlap = pd.merge(
    get_frac_overlap(
        marker_genes_dict,
        marker_genes_highqc.groupby('group')['gene'].apply(list).to_dict(),
    ),
    get_frac_overlap(
        marker_genes_dict,
        marker_genes_HLCAv1.groupby('group')['gene'].apply(list).to_dict(),
    ),
    on='cell_type',
    suffixes=['_highQC', '_HLCAv1'],
).set_index('cell_type')

In [ ]:
marker_overlap['diff'] = marker_overlap['overlap_highQC'] - marker_overlap['overlap_HLCAv1']

In [ ]:
marker_genes_dict['CD4 T cells']

### Supplementary Table 5

In [ ]:
supp_table = marker_overlap.query(
    'missing_highQC != missing_HLCAv1'
).sort_values('diff', ascending=False)
supp_table

In [ ]:
supp_table.to_csv(figure_dir / 'supp_table_markers.tsv', sep='\t')

### Overlap plot

In [ ]:
# Sort by highQC
back_to_back_barplot(
    marker_overlap.query('missing_highQC != missing_HLCAv1').reset_index(),
    group_col='cell_type',
    left_col='overlap_HLCAv1',
    right_col='overlap_highQC',
    title='Overlap with curated markers',
    figsize=(7, 6),
    xlim_left=1,
    xlim_right=1,
)

In [ ]:
# Sort by HLCAv1
back_to_back_barplot(
    marker_overlap.query('missing_highQC != missing_HLCAv1').reset_index(),
    group_col='cell_type',
    left_col='overlap_highQC',
    right_col='overlap_HLCAv1',
    title='Overlap with curated markers',
    figsize=(7, 6),
    xlim_left=1,
    xlim_right=1,
)

In [ ]:
marker_overlap_long = marker_overlap.query('missing_highQC != missing_HLCAv1').reset_index().melt(
    id_vars=['cell_type', 'diff'],
    value_vars=['overlap_HLCAv1', 'overlap_highQC'],
    var_name='Reference',
    value_name='Overlap',
).assign(
    Reference=lambda x: x['Reference'].str.replace('overlap_', '')
)

In [ ]:
order = marker_overlap_long.query('Reference == "highQC"').sort_values('Overlap', ascending=False)['cell_type']
g = sns.catplot(
    marker_overlap_long,
    x='Overlap',
    y='cell_type',
    hue='Reference',
    kind='bar', order=order,
    palette=['steelblue', 'tomato'],
    height=5, aspect=1.0,
    saturation=0.9,
)
g.set(xlabel='', ylabel='', xlim=(0, 1))
g.figure.suptitle('Overlap with published HLCA markers')
g.figure.tight_layout()
sns.move_legend(
    g, 'lower right',
    bbox_to_anchor=(0.95, 0.1),
    title=None,
    frameon=True, facecolor='white', edgecolor='white',
)

plt.savefig(figure_dir / 'Ext2_B_curated_markers.svg')